# LACT_sim ROOT quicklook with pylast

This notebook reads a LACT_sim `lact_event_root_v1` file with `pylast.io.LactEventSource`, checks the event content, and draws the two standard quicklook views used for LACT_sim ROOT validation:

- array/core quicklook: telescope layout, shower core, event direction, telescope pointing, triggered telescope outline
- triggered camera quicklook: camera images for the triggered telescopes in one event

On the server, run it inside the independent pylast environment after loading the same ROOT used to build pylast. Set `ROOT_FILE` below to the ROOT file produced by `run_corsika_trace`.


In [ ]:
from pathlib import Path
import os

# Keep matplotlib/cache writes away from AFS/home quota.
os.environ.setdefault("MPLCONFIGDIR", "/home/lhaaso/huangyiyun/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

# Server default. Change this if you wrote ROOT to another output directory.
ROOT_FILE = Path("/home/lhaaso/huangyiyun/LACT/Sim_program/LACT_sim/run_logs/lact_root_only_full_response/lact_events.root")
if not ROOT_FILE.exists():
    ROOT_FILE = Path("/home/lhaaso/huangyiyun/LACT/Sim_program/LACT_sim/run_logs/lact_root_full_response/lact_events.root")

EVENT_INDEX = 0
MAX_EVENTS = 10
IMAGE_LEVEL = "dl0"

print("ROOT_FILE:", ROOT_FILE)
print("exists:", ROOT_FILE.exists())


In [ ]:
import pylast
from pylast.io import LactEventSource
from pylast.visualize import plot_event_cores, plot_event_cameras, plot_lact_root_quicklook

source = LactEventSource(str(ROOT_FILE), max_events=MAX_EVENTS)
event = source[EVENT_INDEX]
triggered_tels = list(getattr(event.simulation, "triggered_tels", []))

print("event_id:", event.event_id)
print("run_id:", event.run_id)
print("loaded max_events:", getattr(source, "max_events", None))
print("subarray telescope ids:", sorted(source.subarray.tels.keys()))
print("triggered telescope ids:", triggered_tels)
print("DL0 telescope ids:", sorted(event.dl0.tels.keys()))
print("R1 telescope ids:", sorted(event.r1.tels.keys()))


In [ ]:
# Inspect one telescope payload. Prefer a triggered telescope when available.
tel_id = triggered_tels[0] if triggered_tels else sorted(event.dl0.tels.keys())[0]
dl0_camera = event.dl0.tels[tel_id]
r1_camera = event.r1.tels[tel_id]

print("tel_id:", tel_id)
print("DL0 image shape:", dl0_camera.image.shape)
print("DL0 peak_time shape:", dl0_camera.peak_time.shape)
print("R1 waveform shape:", r1_camera.waveform.shape)
print("R1 gain_selection shape:", r1_camera.gain_selection.shape)
print("DL0 total p.e.:", float(dl0_camera.image.sum()))
print("R1 waveform total p.e.:", float(r1_camera.waveform.sum()))


## 1. Array/Core Quicklook

This cell draws the telescope layout in the LACT_sim NE-style array coordinates, the shower core, event arrival direction, telescope pointing direction, and red outlines for triggered telescopes.


In [ ]:
array_result = plot_event_cores(
    root_file=ROOT_FILE,
    event_index=EVENT_INDEX,
    max_events=MAX_EVENTS,
    image_level=IMAGE_LEVEL,
    show_sdp_planes=True,
)

array_result["figure"]


## 2. Triggered Camera Images

This cell draws camera images for the triggered telescopes in the same event. Set `include_non_triggered=True` if you intentionally want to inspect non-triggered telescopes that still have nonzero p.e. in the full ROOT output.


In [ ]:
camera_result = plot_event_cameras(
    root_file=ROOT_FILE,
    event_index=EVENT_INDEX,
    max_events=MAX_EVENTS,
    image_level=IMAGE_LEVEL,
    include_non_triggered=False,
)

camera_result["figure"]


## Optional: Save All Quicklook PNGs


In [ ]:
OUTPUT_DIR = ROOT_FILE.parent / "pylast_visualize"

save_result = plot_lact_root_quicklook(
    root_file=ROOT_FILE,
    output_dir=OUTPUT_DIR,
    event_index=EVENT_INDEX,
    max_events=MAX_EVENTS,
    image_level=IMAGE_LEVEL,
    show=False,
)

for name, path in save_result["paths"].items():
    print(f"{name}: {path}")


## Server Notes

A typical server workflow is:

```bash
cd /home/lhaaso/huangyiyun/LACT/Sim_program/pylast
git pull yun lact_sim

export CONDA_PKGS_DIRS=/home/lhaaso/huangyiyun/conda/pkgs
export MPLCONFIGDIR=/home/lhaaso/huangyiyun/tmp/matplotlib

python -m pip install -e . --no-build-isolation
jupyter lab notebooks/lact_sim_root_quicklook.ipynb
```

If ROOT is not on the default linker path, load/export the same ROOT used during build before launching Jupyter.
